# Forensic Sketch Generation Using GenAI — FastAPI + ngrok Backend
Run each cell in order. After Cell 4 you'll get a public `https://xxxx.ngrok-free.app` URL — paste that into the CRIMINET dashboard.

In [ ]:
# ── CELL 1: Install all dependencies ──────────────────────────────────────
!pip install -q fastapi uvicorn[standard] pyngrok nest-asyncio python-multipart
!pip install -U -q bitsandbytes>=0.46.1 transformers accelerate diffusers safetensors
!pip install -U -q insightface onnxruntime-gpu faiss-cpu opencv-python-headless gdown
!git clone -q https://github.com/zllrunning/face-parsing.PyTorch.git
%cd /content/face-parsing.PyTorch
!mkdir -p res/cp
!gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O res/cp/79999_iter.pth -q
import os
if not os.path.exists('res/cp/79999_iter.pth') or os.path.getsize('res/cp/79999_iter.pth') < 1_000_000:
    print('Falling back to HuggingFace mirror...')
    !wget -q -O res/cp/79999_iter.pth https://huggingface.co/manyotherfunctions/face-parse-bisent/resolve/main/79999_iter.pth
%cd /content/
print('✅ Dependencies installed.')

In [ ]:
# ── CELL 2: Upload fastapi_backend.py ─────────────────────────────────────
# Option A — upload from your machine:
from google.colab import files
print('Upload fastapi_backend.py from your computer:')
uploaded = files.upload()   # select fastapi_backend.py

# Option B — if you saved it to Drive, copy it instead:
# !cp /content/drive/MyDrive/criminet/fastapi_backend.py /content/fastapi_backend.py

import os
assert os.path.exists('/content/fastapi_backend.py'), 'Upload failed!'
print('✅ fastapi_backend.py ready.')

In [ ]:
# ── CELL 3: Load ML models (takes ~3-5 min on first run) ──────────────────
import sys
sys.path.insert(0, '/content')
from fastapi_backend import load_models
load_models()
print('✅ Models ready.')

In [ ]:
# ── CELL 4: Mount Drive + build FAISS index ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# ← Change this to your Drive folder name
DRIVE_DB_PATH = '/content/drive/MyDrive/idoc_front_1000'
LOCAL_DB_PATH = '/content/local_mugshots'

if os.path.exists(DRIVE_DB_PATH):
    print('🚀 Copying database to local SSD...')
    if os.path.exists(LOCAL_DB_PATH): shutil.rmtree(LOCAL_DB_PATH)
    shutil.copytree(DRIVE_DB_PATH, LOCAL_DB_PATH)
    print(f'✅ Copied to {LOCAL_DB_PATH}')

    # Build the FAISS index via the backend directly
    from fastapi_backend import build_index, BuildIndexRequest
    result = build_index(BuildIndexRequest(folder_path=LOCAL_DB_PATH))
    print(f'✅ Index built: {result["indexed"]} faces indexed, {result["failed"]} failed.')
else:
    print(f'⚠️  Drive path not found: {DRIVE_DB_PATH}')
    print('   You can build the index later from the dashboard using the correct path.')

In [ ]:
# ── CELL 5: Start FastAPI server + expose via ngrok ───────────────────────
#
# IMPORTANT: Get a free ngrok authtoken at https://dashboard.ngrok.com
# Then paste it below.

NGROK_AUTHTOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'   # ← replace this

import nest_asyncio, uvicorn, threading
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Authenticate ngrok
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Kill any existing ngrok tunnels
ngrok.kill()

# Open tunnel on port 8000
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url

print('\n' + '='*60)
print(f'  🌐 Forensic Sketch Generator PUBLIC URL:')
print(f'  {public_url}')
print('='*60)
print('  Paste this URL into the Forensic Sketch Generator dashboard and click CONNECT')
print('='*60 + '\n')

# Import the already-initialised app (models already in memory)
import sys
sys.path.insert(0, '/content')
from fastapi_backend import app as fastapi_app

# Start uvicorn in a background thread so this cell doesn't block
def run_server():
    uvicorn.run(fastapi_app, host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()
print('✅ Server running. Keep this notebook open.')

In [ ]:
# ── CELL 6 (optional): Quick health-check ─────────────────────────────────
import requests, json
r = requests.get(f'{public_url}/health')
print(json.dumps(r.json(), indent=2))